In [ ]:
import numpy as np
import pickle

In [25]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

In [26]:
class TwoLayerNet:
    def __init__(self, input_size, hidden_size, output_size, seed):
        rng = np.random.default_rng(seed)
        self.params = {
            'W1': rng.normal(0, 1 / np.sqrt(input_size), (input_size, hidden_size)),
            'b1': np.zeros(hidden_size),
            'W2': rng.normal(0, 1 / np.sqrt(hidden_size), (hidden_size, output_size)),
            'b2': np.zeros(output_size)
        }

    def scores(self, x):
        p = self.params
        hidden = sigmoid(x @ p['W1'] + p['b1'])
        return hidden @ p['W2'] + p['b2']

    def predict(self, x):
        scores = self.scores(x)
        scores -= scores.max(axis=1, keepdims=True)
        probabilities = np.exp(scores)
        return probabilities / probabilities.sum(axis=1, keepdims=True)

    def loss(self, x, y):
        scores = self.scores(x)
        scores -= scores.max(axis=1, keepdims=True)
        return np.mean(np.log(np.exp(scores).sum(axis=1)) - scores[np.arange(len(y)), y])

    def accuracy(self, x, y):
        return np.mean(self.predict(x).argmax(axis=1) == y)

    def numerical_gradient(self, x, y, h=1e-5):
        gradients = {}
        for name, parameter in self.params.items():
            gradient = np.zeros_like(parameter)
            for index in np.ndindex(parameter.shape):
                original = parameter[index]
                try:
                    parameter[index] = original + h
                    loss_plus = self.loss(x, y)
                    parameter[index] = original - h
                    loss_minus = self.loss(x, y)
                finally:
                    parameter[index] = original
                gradient[index] = (loss_plus - loss_minus) / (2 * h)
            gradients[name] = gradient
        return gradients

    def train(network, X, y, epochs, batch_size, learning_rate, print_every=1):
        rng = np.random.default_rng(42)
        for epoch in range(1, epochs + 1):
            order = rng.permutation(len(y))
            for start in range(0, len(y), batch_size):
                batch = order[start:start + batch_size]
                gradients = network.numerical_gradient(X[batch], y[batch])
                for name in network.params:
                    network.params[name] -= learning_rate * gradients[name]
            if epoch % print_every == 0 or epoch == epochs:
                print(f"Epoch {epoch}/{epochs} | Loss: {network.loss(X, y):.4f} | "
                    f"Training accuracy: {network.accuracy(X, y):.2%}", flush=True)

In [27]:
iris = np.genfromtxt("data/iris/iris.data", delimiter=",", dtype=str)
X_iris = iris[:, :4].astype(np.float64)
iris_classes, y_iris = np.unique(iris[:, 4], return_inverse=True)

rng = np.random.default_rng(42)
train_indices, test_indices = [], []
for label in np.unique(y_iris):
    indices = rng.permutation(np.flatnonzero(y_iris == label))
    split = int(0.8 * len(indices))
    train_indices.extend(indices[:split])
    test_indices.extend(indices[split:])
train_indices = np.array(train_indices)
test_indices = np.array(test_indices)

iris_mean = X_iris[train_indices].mean(axis=0)
iris_scale = X_iris[train_indices].std(axis=0)
iris_scale = np.where(iris_scale == 0, 1, iris_scale)
X_iris_train = (X_iris[train_indices] - iris_mean) / iris_scale
y_iris_train = y_iris[train_indices]
print(f"Iris: {len(train_indices)} training samples, {len(test_indices)} test samples")


Iris: 120 training samples, 30 test samples


In [28]:
iris_net = TwoLayerNet(4, 10, 3, seed=27)

iris_net.train(X_iris_train, y_iris_train,
      epochs=1000, batch_size=len(y_iris_train), learning_rate=0.1,
      print_every=100)

Epoch 100/1000 | Loss: 0.5556 | Training accuracy: 86.67%
Epoch 200/1000 | Loss: 0.4259 | Training accuracy: 89.17%
Epoch 300/1000 | Loss: 0.3588 | Training accuracy: 89.17%
Epoch 400/1000 | Loss: 0.3116 | Training accuracy: 92.50%
Epoch 500/1000 | Loss: 0.2737 | Training accuracy: 92.50%
Epoch 600/1000 | Loss: 0.2414 | Training accuracy: 95.00%
Epoch 700/1000 | Loss: 0.2136 | Training accuracy: 95.00%
Epoch 800/1000 | Loss: 0.1901 | Training accuracy: 95.83%
Epoch 900/1000 | Loss: 0.1708 | Training accuracy: 95.00%
Epoch 1000/1000 | Loss: 0.1550 | Training accuracy: 95.00%


In [30]:
iris_model_file = f"lisk_iris_nn_model.pkl"
with open(iris_model_file, "wb") as file:
    pickle.dump({"network": iris_net, "mean": iris_mean, "scale": iris_scale,
                 "classes": iris_classes, "train_indices": train_indices,
                 "test_indices": test_indices}, file)

with open(iris_model_file, "rb") as file:
    iris_saved = pickle.load(file)

iris_test_indices = iris_saved["test_indices"]
X_iris_test = (X_iris[iris_test_indices] - iris_saved["mean"]) / iris_saved["scale"]
y_iris_test = y_iris[iris_test_indices]
iris_predictions = iris_saved["network"].predict(X_iris_test).argmax(axis=1)

print(f"{'Sample':<8}{'Actual':<20}{'Predicted':<20}Result")
for sample, actual, predicted in zip(iris_test_indices, y_iris_test, iris_predictions):
    result = "Correct" if actual == predicted else "Incorrect"
    print(f"{sample:<8}{iris_saved['classes'][actual]:<20}"
          f"{iris_saved['classes'][predicted]:<20}{result}")
print(f"Iris test accuracy: {np.mean(iris_predictions == y_iris_test):.2%}")
print("Saved:", iris_model_file)

Sample  Actual              Predicted           Result
0       Iris-setosa         Iris-setosa         Correct
12      Iris-setosa         Iris-setosa         Correct
14      Iris-setosa         Iris-setosa         Correct
48      Iris-setosa         Iris-setosa         Correct
2       Iris-setosa         Iris-setosa         Correct
36      Iris-setosa         Iris-setosa         Correct
33      Iris-setosa         Iris-setosa         Correct
1       Iris-setosa         Iris-setosa         Correct
13      Iris-setosa         Iris-setosa         Correct
8       Iris-setosa         Iris-setosa         Correct
79      Iris-versicolor     Iris-versicolor     Correct
81      Iris-versicolor     Iris-versicolor     Correct
51      Iris-versicolor     Iris-versicolor     Correct
99      Iris-versicolor     Iris-versicolor     Correct
67      Iris-versicolor     Iris-versicolor     Correct
89      Iris-versicolor     Iris-versicolor     Correct
91      Iris-versicolor     Iris-versicolor     C

In [32]:
with np.load("data/mnist.npz", allow_pickle=False) as data:
    X_mnist_train = data["x_train"].reshape(-1, 784).astype(np.float64) / 255.0
    y_mnist_train = data["y_train"]
    X_mnist_test_raw = data["x_test"]
    y_mnist_test = data["y_test"]
print(f"MNIST: {len(y_mnist_train)} training samples, {len(y_mnist_test)} test samples")


MNIST: 60000 training samples, 10000 test samples


In [39]:
mnist_net = TwoLayerNet(784, 10, 10, seed=27)

mnist_net.train(X_mnist_train, y_mnist_train, epochs=3, batch_size=128, learning_rate=0.1)

Epoch 1/3 | Loss: 1.0173 | Training accuracy: 79.52%
Epoch 2/3 | Loss: 0.6525 | Training accuracy: 85.19%
Epoch 3/3 | Loss: 0.5268 | Training accuracy: 86.95%


In [40]:
mnist_model_file = f"lisk_mnist_nn_model.pkl"
with open(mnist_model_file, "wb") as file:
    pickle.dump({"network": mnist_net, "pixel_scale": 255.0}, file)

with open(mnist_model_file, "rb") as file:
    mnist_saved = pickle.load(file)

X_mnist_test = (X_mnist_test_raw.reshape(-1, 784).astype(np.float64)
                / mnist_saved["pixel_scale"])
mnist_predictions = mnist_saved["network"].predict(X_mnist_test).argmax(axis=1)
print(f"MNIST test accuracy: {np.mean(mnist_predictions == y_mnist_test):.2%}")
print(f"{'Sample':<8}{'Actual':<8}{'Predicted':<12}Result")
for sample in range(20):
    actual, predicted = y_mnist_test[sample], mnist_predictions[sample]
    result = "Correct" if actual == predicted else "Incorrect"
    print(f"{sample:<8}{actual:<8}{predicted:<12}{result}")
print("Saved:", mnist_model_file)

MNIST test accuracy: 87.86%
Sample  Actual  Predicted   Result
0       7       7           Correct
1       2       2           Correct
2       1       1           Correct
3       0       0           Correct
4       4       4           Correct
5       1       1           Correct
6       4       4           Correct
7       9       9           Correct
8       5       4           Incorrect
9       9       9           Correct
10      0       0           Correct
11      6       6           Correct
12      9       9           Correct
13      0       0           Correct
14      1       1           Correct
15      5       5           Correct
16      9       9           Correct
17      7       7           Correct
18      3       3           Correct
19      4       4           Correct
Saved: lisk_mnist_nn_model.pkl
